In [2]:
import pandas as pd
import numpy as np
import os
import ast

In [3]:
# LOAD RESULTS 

run_id = "dbd491bd2e61448fa3f3b4b083af11ac"
path_to_results = os.path.join('..',
            'output', run_id)
results = pd.read_csv(os.path.join(path_to_results, "03_co2_emission_table2_w_query_responses.csv"),
                dtype={
                    'extracted_scope_from_llm_orig': str,
                    'extracted_scope_from_llm': str,
                    'page_number_used_by_llm': str,
                    'page_number_to_llm': str
            })
results['page_numbers_tried_by_llm'] = results['page_numbers_tried_by_llm'].apply(ast.literal_eval)

In [42]:
path_to_ground_truth = '../data/evaluation_dataset/gist_2025.csv'

ground_truth = pd.read_csv(path_to_ground_truth, dtype={
    'scope': str, 'page': str}, decimal=',')
ground_truth_subset = ground_truth[[
    'report_name', 'scope', 'year', 'value', 'unit', 'page', 'metric_name', 'display_type']]
ground_truth_subset = ground_truth_subset.assign(
    ms_comment_man=None)

rename_columns_map = {
    'report_name': 'ReportName',
    'scope': 'scope_man',
    'year': 'year_man',
    'value': 'value_man',
    'unit': 'unit_man',
    'page': 'page_man',
    'metric_name': 'val_name_man',
    'display_type': 'type_man',
}
ground_truth_subset = ground_truth_subset.rename(
    columns=rename_columns_map)

ground_truth = ground_truth_subset.copy()

ground_truth['value_man'] = pd.to_numeric(
    ground_truth['value_man'].str.replace(',', '.'), errors='coerce')
ground_truth['year_man'] = pd.to_numeric(
    ground_truth['year_man'], errors='coerce')

In [43]:
len(ground_truth)


5644

## Test with 3 reports

In [6]:
# Test with only 3 reports
selected_reports = [
    'Allianz_2022_report.pdf', # double values
    'addtech_2022_report.pdf', # all correct
    'Daimler_2020_report.pdf', # fails to find all non-NA correct values
]

In [33]:
ground_truth_selected = ground_truth[ground_truth['ReportName'].isin(selected_reports)]
len(ground_truth_selected)

131

In [75]:
ground_truth_selected

,ReportName,scope_man,year_man,value_man,unit_man,page_man,val_name_man,type_man,ms_comment_man
0,Allianz_2022_report.pdf,1,2013,NaN,NaN,NaN,NaN,NaN,None
1,Allianz_2022_report.pdf,1,2014,NaN,NaN,NaN,NaN,NaN,None
2,Allianz_2022_report.pdf,1,2015,NaN,NaN,NaN,NaN,NaN,None
3,Allianz_2022_report.pdf,1,2016,NaN,NaN,NaN,NaN,NaN,None
4,Allianz_2022_report.pdf,1,2017,NaN,NaN,NaN,NaN,NaN,None
...,...,...,...,...,...,...,...,...,...
208,addtech_2022_report.pdf,3,2018,NaN,NaN,NaN,NaN,NaN,None
209,addtech_2022_report.pdf,3,2019,23132.0,tonnes CO2e,73,SCOPE 3/Emissions of greenhouse gases - Scope 3,Table,None
210,addtech_2022_report.pdf,3,2020,19860.0,tonnes CO2e,73,SCOPE 4/Emissions of greenhouse gases - Scope 3,Table,None
211,addtech_2022_report.pdf,3,2021,22961.0,tonnes CO2e,73,SCOPE 5/Emissions of greenhouse gases - Scope 3,Table,None


In [44]:
run_id = "1d23780d6b4348d6a713753bc10801a0"
path_to_results = os.path.join('..', 'output', run_id)
results = pd.read_csv(os.path.join(
            path_to_results, "03_co2_emission_table2_w_query_responses.csv"),
            dtype={
                'extracted_scope_from_llm_orig': str,
                'extracted_scope_from_llm': str,
                'page_number_used_by_llm': str,
                'page_number_to_llm': str
        })
results['page_numbers_tried_by_llm'] = results['page_numbers_tried_by_llm'].apply(
            ast.literal_eval)
len(results)

55617

In [35]:
results_selected = results[results['report_name_short'].isin(selected_reports)]
len(results_selected)

1138

In [37]:
# create additional variables "automatic_extraction_tried" and "page_numbers_tried_by_llm" in ground_truth
grtruth_extended_selected = results_selected[[
    "report_name_short", "automatic_extraction_tried",  "page_numbers_tried_by_llm"]]
grtruth_extended_selected.loc[:, 'page_numbers_tried_by_llm'] = grtruth_extended_selected.loc[:, 'page_numbers_tried_by_llm'].apply(
    tuple)
grtruth_extended_selected = grtruth_extended_selected.groupby(["report_name_short", "page_numbers_tried_by_llm"])[
    'automatic_extraction_tried'].apply(lambda group: group.any(skipna=True)).reset_index()
grtruth_extended_selected.loc[:, 'page_numbers_tried_by_llm'] = grtruth_extended_selected.loc[:, 'page_numbers_tried_by_llm'].apply(
    list)

grtruth_extended_selected = pd.merge(ground_truth_selected, grtruth_extended_selected, how="left",
                            left_on="ReportName", right_on="report_name_short")
grtruth_extended_selected = grtruth_extended_selected.drop('report_name_short', axis=1)
grtruth_extended_selected.loc[grtruth_extended_selected['automatic_extraction_tried'].isna(
), 'automatic_extraction_tried'] = False

len(grtruth_extended_selected)

131

In [38]:
# 2. Strip irrelevant rows from LLM output
# it is not obvious how to keep rows that have "helpful" data/ extracted values.
# at this step we lose information about page numbers passed to the LLM
# if no valuable information was returned.
#  Option 1:
tiny_results_selected = results[results['extracted_value_from_llm'].notnull(
)]

# Option 2:
# tiny_results = co2_emission_table2_w_query_responses[~(
#            (co2_emission_table2_w_query_responses['extracted_value_from_llm_orig'] == "Not specified") |
#            (co2_emission_table2_w_query_responses['extracted_value_from_llm_orig'] == "Nothing extracted. No Regex match")
# )]

# 3. Left-merge
tiny_results_selected = tiny_results_selected.drop(
    'automatic_extraction_tried', axis=1)
tiny_results_selected = tiny_results_selected.drop(
    'page_numbers_tried_by_llm', axis=1)

len(tiny_results_selected)

1130

# Notes on Left merge on ReportName, Scope and Year

Difference between ground_truth with selection and merged dataset = 144 - 131 = 13 rows

Reasons:
- Results df contains multiple values for same scope-year combination
e.g. Allianz 2lb for 2022: one value with tCO2e and one value with ktCO2e, different pages
- Ground truth contains multiple values for same scope-year combination 
e.g. Allianz 2mb for 2022: one value with tCO2e and one value with ktCO2e, different pages

Solution: Merge on page

In [39]:
# LEFT MERGE
left_merged_results = pd.merge(ground_truth_selected, tiny_results_selected, how="left",
                                  left_on=["ReportName",
                                           "scope_man", "year_man"],
                                  right_on=["report_name_short", "extracted_scope_from_llm",
                                            "extracted_year_from_llm"], indicator=True)
len(left_merged_results)

144

In [14]:
left_merged_results['_merge'].value_counts()

_merge
left_only     79
both          65
right_only     0
Name: count, dtype: int64

In [15]:
left_merged_results

,ReportName,scope_man,year_man,value_man,unit_man,page_man,val_name_man,type_man,ms_comment_man,page_numbers_tried_by_llm,...,extracted_value_from_llm,raw_llm_response,page_number_used_by_llm,report_name,page_number_to_llm,page_retrieval_scores,page_texts_to_llm,text_response_from_llm,report_name_short,_merge
0,Allianz_2022_report.pdf,1,2013,NaN,NaN,NaN,NaN,NaN,None,"[47, 48, 49, 78, 79, 80, 90, 91, 92]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1,Allianz_2022_report.pdf,1,2014,NaN,NaN,NaN,NaN,NaN,None,"[47, 48, 49, 78, 79, 80, 90, 91, 92]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2,Allianz_2022_report.pdf,1,2015,NaN,NaN,NaN,NaN,NaN,None,"[47, 48, 49, 78, 79, 80, 90, 91, 92]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
3,Allianz_2022_report.pdf,1,2016,NaN,NaN,NaN,NaN,NaN,None,"[47, 48, 49, 78, 79, 80, 90, 91, 92]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
4,Allianz_2022_report.pdf,1,2017,NaN,NaN,NaN,NaN,NaN,None,"[47, 48, 49, 78, 79, 80, 90, 91, 92]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,addtech_2022_report.pdf,3,2018,NaN,NaN,NaN,NaN,NaN,None,"[68, 69, 70, 72, 73, 74]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
140,addtech_2022_report.pdf,3,2019,23132.0,tonnes CO2e,73,SCOPE 3/Emissions of greenhouse gases - Scope 3,Table,None,"[68, 69, 70, 72, 73, 74]",...,23132.0,"```json\n{\n ""KPI_Entries"": [\n {\n ""...",73,./data/pdfs/addtech_2022_report.pdf,73,0.807174,144 ADDTECH ANNUAL REPORT 2021/2022 ADDTEC...,"0 ```json\n{\n ""KPI_Entries"": [\n {\n ...",addtech_2022_report.pdf,both
141,addtech_2022_report.pdf,3,2020,19860.0,tonnes CO2e,73,SCOPE 4/Emissions of greenhouse gases - Scope 3,Table,None,"[68, 69, 70, 72, 73, 74]",...,19860.0,"```json\n{\n ""KPI_Entries"": [\n {\n ""...",73,./data/pdfs/addtech_2022_report.pdf,73,0.807174,144 ADDTECH ANNUAL REPORT 2021/2022 ADDTEC...,"0 ```json\n{\n ""KPI_Entries"": [\n {\n ...",addtech_2022_report.pdf,both
142,addtech_2022_report.pdf,3,2021,22961.0,tonnes CO2e,73,SCOPE 5/Emissions of greenhouse gases - Scope 3,Table,None,"[68, 69, 70, 72, 73, 74]",...,22961.0,"```json\n{\n ""KPI_Entries"": [\n {\n ""...",73,./data/pdfs/addtech_2022_report.pdf,73,0.807174,144 ADDTECH ANNUAL REPORT 2021/2022 ADDTEC...,"0 ```json\n{\n ""KPI_Entries"": [\n {\n ...",addtech_2022_report.pdf,both


# Notes on Outer merge on ReportName, Scope and Year

No difference to left merge because all scope-year combinations already present in ground truth

In [ ]:
# OUTER MERGE
outer_merged_results = pd.merge(grtruth_extended_selected, tiny_results_selected, how="outer",
                                  left_on=["ReportName",
                                           "scope_man", "year_man"],
                                  right_on=["report_name_short", "extracted_scope_from_llm",
                                            "extracted_year_from_llm"], indicator=True)
len(outer_merged_results)

144

In [17]:
outer_merged_results['_merge'].value_counts()

_merge
left_only     79
both          65
right_only     0
Name: count, dtype: int64

In [18]:
outer_merged_results

,ReportName,scope_man,year_man,value_man,unit_man,page_man,val_name_man,type_man,ms_comment_man,page_numbers_tried_by_llm,...,extracted_value_from_llm,raw_llm_response,page_number_used_by_llm,report_name,page_number_to_llm,page_retrieval_scores,page_texts_to_llm,text_response_from_llm,report_name_short,_merge
0,Allianz_2022_report.pdf,1,2013,NaN,NaN,NaN,NaN,NaN,None,"[47, 48, 49, 78, 79, 80, 90, 91, 92]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1,Allianz_2022_report.pdf,1,2014,NaN,NaN,NaN,NaN,NaN,None,"[47, 48, 49, 78, 79, 80, 90, 91, 92]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2,Allianz_2022_report.pdf,1,2015,NaN,NaN,NaN,NaN,NaN,None,"[47, 48, 49, 78, 79, 80, 90, 91, 92]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
3,Allianz_2022_report.pdf,1,2016,NaN,NaN,NaN,NaN,NaN,None,"[47, 48, 49, 78, 79, 80, 90, 91, 92]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
4,Allianz_2022_report.pdf,1,2017,NaN,NaN,NaN,NaN,NaN,None,"[47, 48, 49, 78, 79, 80, 90, 91, 92]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,addtech_2022_report.pdf,3,2018,NaN,NaN,NaN,NaN,NaN,None,"[68, 69, 70, 72, 73, 74]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
140,addtech_2022_report.pdf,3,2019,23132.0,tonnes CO2e,73,SCOPE 3/Emissions of greenhouse gases - Scope 3,Table,None,"[68, 69, 70, 72, 73, 74]",...,23132.0,"```json\n{\n ""KPI_Entries"": [\n {\n ""...",73,./data/pdfs/addtech_2022_report.pdf,73,0.807174,144 ADDTECH ANNUAL REPORT 2021/2022 ADDTEC...,"0 ```json\n{\n ""KPI_Entries"": [\n {\n ...",addtech_2022_report.pdf,both
141,addtech_2022_report.pdf,3,2020,19860.0,tonnes CO2e,73,SCOPE 4/Emissions of greenhouse gases - Scope 3,Table,None,"[68, 69, 70, 72, 73, 74]",...,19860.0,"```json\n{\n ""KPI_Entries"": [\n {\n ""...",73,./data/pdfs/addtech_2022_report.pdf,73,0.807174,144 ADDTECH ANNUAL REPORT 2021/2022 ADDTEC...,"0 ```json\n{\n ""KPI_Entries"": [\n {\n ...",addtech_2022_report.pdf,both
142,addtech_2022_report.pdf,3,2021,22961.0,tonnes CO2e,73,SCOPE 5/Emissions of greenhouse gases - Scope 3,Table,None,"[68, 69, 70, 72, 73, 74]",...,22961.0,"```json\n{\n ""KPI_Entries"": [\n {\n ""...",73,./data/pdfs/addtech_2022_report.pdf,73,0.807174,144 ADDTECH ANNUAL REPORT 2021/2022 ADDTEC...,"0 ```json\n{\n ""KPI_Entries"": [\n {\n ...",addtech_2022_report.pdf,both


In [19]:
# difference between left and outer merge
mask = ~outer_merged_results.isin(left_merged_results.to_dict(orient='list')).all(axis=1)
result = outer_merged_results[mask]
result

,ReportName,scope_man,year_man,value_man,unit_man,page_man,val_name_man,type_man,ms_comment_man,page_numbers_tried_by_llm,...,extracted_value_from_llm,raw_llm_response,page_number_used_by_llm,report_name,page_number_to_llm,page_retrieval_scores,page_texts_to_llm,text_response_from_llm,report_name_short,_merge


# Notes on Left merge on ReportName, Scope, Year and Page

- Same length as ground truth filtered by reports
- no additional rows

In [ ]:
# Left merge with page numbers
left_w_page_merged_results = pd.merge(grtruth_extended_selected, tiny_results_selected, how="left",
                                  left_on=["ReportName",
                                           "scope_man", "year_man", "page_man"],
                                  right_on=["report_name_short", "extracted_scope_from_llm",
                                            "extracted_year_from_llm", "page_number_used_by_llm"], indicator=True)
len(left_w_page_merged_results)

131

In [21]:
left_w_page_merged_results['_merge'].value_counts()

_merge
left_only     90
both          41
right_only     0
Name: count, dtype: int64

In [22]:
left_w_page_merged_results

,ReportName,scope_man,year_man,value_man,unit_man,page_man,val_name_man,type_man,ms_comment_man,page_numbers_tried_by_llm,...,extracted_value_from_llm,raw_llm_response,page_number_used_by_llm,report_name,page_number_to_llm,page_retrieval_scores,page_texts_to_llm,text_response_from_llm,report_name_short,_merge
0,Allianz_2022_report.pdf,1,2013,NaN,NaN,NaN,NaN,NaN,None,"[47, 48, 49, 78, 79, 80, 90, 91, 92]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1,Allianz_2022_report.pdf,1,2014,NaN,NaN,NaN,NaN,NaN,None,"[47, 48, 49, 78, 79, 80, 90, 91, 92]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2,Allianz_2022_report.pdf,1,2015,NaN,NaN,NaN,NaN,NaN,None,"[47, 48, 49, 78, 79, 80, 90, 91, 92]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
3,Allianz_2022_report.pdf,1,2016,NaN,NaN,NaN,NaN,NaN,None,"[47, 48, 49, 78, 79, 80, 90, 91, 92]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
4,Allianz_2022_report.pdf,1,2017,NaN,NaN,NaN,NaN,NaN,None,"[47, 48, 49, 78, 79, 80, 90, 91, 92]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126,addtech_2022_report.pdf,3,2018,NaN,NaN,NaN,NaN,NaN,None,"[68, 69, 70, 72, 73, 74]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
127,addtech_2022_report.pdf,3,2019,23132.0,tonnes CO2e,73,SCOPE 3/Emissions of greenhouse gases - Scope 3,Table,None,"[68, 69, 70, 72, 73, 74]",...,23132.0,"```json\n{\n ""KPI_Entries"": [\n {\n ""...",73,./data/pdfs/addtech_2022_report.pdf,73,0.807174,144 ADDTECH ANNUAL REPORT 2021/2022 ADDTEC...,"0 ```json\n{\n ""KPI_Entries"": [\n {\n ...",addtech_2022_report.pdf,both
128,addtech_2022_report.pdf,3,2020,19860.0,tonnes CO2e,73,SCOPE 4/Emissions of greenhouse gases - Scope 3,Table,None,"[68, 69, 70, 72, 73, 74]",...,19860.0,"```json\n{\n ""KPI_Entries"": [\n {\n ""...",73,./data/pdfs/addtech_2022_report.pdf,73,0.807174,144 ADDTECH ANNUAL REPORT 2021/2022 ADDTEC...,"0 ```json\n{\n ""KPI_Entries"": [\n {\n ...",addtech_2022_report.pdf,both
129,addtech_2022_report.pdf,3,2021,22961.0,tonnes CO2e,73,SCOPE 5/Emissions of greenhouse gases - Scope 3,Table,None,"[68, 69, 70, 72, 73, 74]",...,22961.0,"```json\n{\n ""KPI_Entries"": [\n {\n ""...",73,./data/pdfs/addtech_2022_report.pdf,73,0.807174,144 ADDTECH ANNUAL REPORT 2021/2022 ADDTEC...,"0 ```json\n{\n ""KPI_Entries"": [\n {\n ...",addtech_2022_report.pdf,both


# Notes on Outer merge with ReportName, Scope, Year and Page

Difference between left and outer merge: 9 rows
- these are rows from results where scope-year combination exists in ground truth, but connected to different page: e.g. Allianz 2lb 2022 0.14ktCO2e
- in left merge with page they are just ignored
- in left/outer merge without page they are added as additional row for 2lb 2022
- in outer merge with page they are added as additional row but with missing values for ground truth ==> making evaluation rather impractical

In [ ]:
# Outer merge with page numbers
outer_w_page_merged_results = pd.merge(grtruth_extended_selected, tiny_results_selected, how="outer",
                                  left_on=["ReportName",
                                           "scope_man", "year_man", "page_man"],
                                  right_on=["report_name_short", "extracted_scope_from_llm",
                                            "extracted_year_from_llm", "page_number_used_by_llm"], indicator=True)
len(outer_w_page_merged_results)

140

In [24]:
outer_w_page_merged_results['_merge'].value_counts()

_merge
left_only     90
both          41
right_only     9
Name: count, dtype: int64

In [25]:
outer_w_page_merged_results

,ReportName,scope_man,year_man,value_man,unit_man,page_man,val_name_man,type_man,ms_comment_man,page_numbers_tried_by_llm,...,extracted_value_from_llm,raw_llm_response,page_number_used_by_llm,report_name,page_number_to_llm,page_retrieval_scores,page_texts_to_llm,text_response_from_llm,report_name_short,_merge
0,Allianz_2022_report.pdf,1,2013.0,NaN,NaN,NaN,NaN,NaN,None,"[47, 48, 49, 78, 79, 80, 90, 91, 92]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1,Allianz_2022_report.pdf,1,2014.0,NaN,NaN,NaN,NaN,NaN,None,"[47, 48, 49, 78, 79, 80, 90, 91, 92]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2,Allianz_2022_report.pdf,1,2015.0,NaN,NaN,NaN,NaN,NaN,None,"[47, 48, 49, 78, 79, 80, 90, 91, 92]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
3,Allianz_2022_report.pdf,1,2016.0,NaN,NaN,NaN,NaN,NaN,None,"[47, 48, 49, 78, 79, 80, 90, 91, 92]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
4,Allianz_2022_report.pdf,1,2017.0,NaN,NaN,NaN,NaN,NaN,None,"[47, 48, 49, 78, 79, 80, 90, 91, 92]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,addtech_2022_report.pdf,3,2019.0,23132.0,tonnes CO2e,73,SCOPE 3/Emissions of greenhouse gases - Scope 3,Table,None,"[68, 69, 70, 72, 73, 74]",...,23132.0,"```json\n{\n ""KPI_Entries"": [\n {\n ""...",73,./data/pdfs/addtech_2022_report.pdf,73,0.807174,144 ADDTECH ANNUAL REPORT 2021/2022 ADDTEC...,"0 ```json\n{\n ""KPI_Entries"": [\n {\n ...",addtech_2022_report.pdf,both
136,addtech_2022_report.pdf,3,2020.0,19860.0,tonnes CO2e,73,SCOPE 4/Emissions of greenhouse gases - Scope 3,Table,None,"[68, 69, 70, 72, 73, 74]",...,19860.0,"```json\n{\n ""KPI_Entries"": [\n {\n ""...",73,./data/pdfs/addtech_2022_report.pdf,73,0.807174,144 ADDTECH ANNUAL REPORT 2021/2022 ADDTEC...,"0 ```json\n{\n ""KPI_Entries"": [\n {\n ...",addtech_2022_report.pdf,both
137,addtech_2022_report.pdf,3,2021.0,22961.0,tonnes CO2e,73,SCOPE 5/Emissions of greenhouse gases - Scope 3,Table,None,"[68, 69, 70, 72, 73, 74]",...,22961.0,"```json\n{\n ""KPI_Entries"": [\n {\n ""...",73,./data/pdfs/addtech_2022_report.pdf,73,0.807174,144 ADDTECH ANNUAL REPORT 2021/2022 ADDTEC...,"0 ```json\n{\n ""KPI_Entries"": [\n {\n ...",addtech_2022_report.pdf,both
138,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5840.0,"```json\n{\n ""KPI_Entries"": [\n {\n ""...",74,./data/pdfs/addtech_2022_report.pdf,74,0.771676,146 ADDTECH ANNUAL REPORT 2021/2022 ADDTECH...,"0 ```json\n{\n ""KPI_Entries"": [\n {\n ...",addtech_2022_report.pdf,right_only


In [31]:
# difference between left and outer merge
mask = ~outer_w_page_merged_results.isin(left_w_page_merged_results.to_dict(orient='list')).all(axis=1)
result = outer_w_page_merged_results[mask]
result

,ReportName,scope_man,year_man,value_man,unit_man,page_man,val_name_man,type_man,ms_comment_man,page_numbers_tried_by_llm,...,extracted_value_from_llm,raw_llm_response,page_number_used_by_llm,report_name,page_number_to_llm,page_retrieval_scores,page_texts_to_llm,text_response_from_llm,report_name_short,_merge
23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.14,"```json\n{\n ""KPI_Entries"": [\n {\n ""...",92,./data/pdfs/Allianz_2022_report.pdf,92,0.796353,01 Introduction \nand strategy02 Measuring and...,"0 ```json\n{\n ""KPI_Entries"": [\n {\n ...",Allianz_2022_report.pdf,right_only
47,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,56316.00,"```json\n{\n ""KPI_Entries"": [\n {\n ""...",79,./data/pdfs/Allianz_2022_report.pdf,79,0.815491,01 Introduction \nand strategy 02 Measuring an...,"0 ```json\n{\n ""KPI_Entries"": [\n {\n ...",Allianz_2022_report.pdf,right_only
50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,39570.00,"```json\n{\n ""KPI_Entries"": [\n {\n ""...",79,./data/pdfs/Allianz_2022_report.pdf,79,0.815491,01 Introduction \nand strategy 02 Measuring an...,"0 ```json\n{\n ""KPI_Entries"": [\n {\n ...",Allianz_2022_report.pdf,right_only
53,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,74339.00,"```json\n{\n ""KPI_Entries"": [\n {\n ""...",79,./data/pdfs/Allianz_2022_report.pdf,79,0.815491,01 Introduction \nand strategy 02 Measuring an...,"0 ```json\n{\n ""KPI_Entries"": [\n {\n ...",Allianz_2022_report.pdf,right_only
92,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,49.70,"```json\n{\n ""KPI_Entries"": [\n {\n ""...",50,./data/pdfs/Daimler_2020_report.pdf,50,0.834293,REPORTING | CLIMATE PROTECTION & AIR QUALITY50...,"0 ```json\n{\n ""KPI_Entries"": [\n {\n ...",Daimler_2020_report.pdf,right_only
105,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1460.00,"```json\n{\n ""KPI_Entries"": [\n {\n ""...",74,./data/pdfs/addtech_2022_report.pdf,74,0.771676,146 ADDTECH ANNUAL REPORT 2021/2022 ADDTECH...,"0 ```json\n{\n ""KPI_Entries"": [\n {\n ...",addtech_2022_report.pdf,right_only
116,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,820.00,"```json\n{\n ""KPI_Entries"": [\n {\n ""...",74,./data/pdfs/addtech_2022_report.pdf,74,0.771676,146 ADDTECH ANNUAL REPORT 2021/2022 ADDTECH...,"0 ```json\n{\n ""KPI_Entries"": [\n {\n ...",addtech_2022_report.pdf,right_only
127,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,730.00,"```json\n{\n ""KPI_Entries"": [\n {\n ""...",74,./data/pdfs/addtech_2022_report.pdf,74,0.771676,146 ADDTECH ANNUAL REPORT 2021/2022 ADDTECH...,"0 ```json\n{\n ""KPI_Entries"": [\n {\n ...",addtech_2022_report.pdf,right_only
138,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5840.00,"```json\n{\n ""KPI_Entries"": [\n {\n ""...",74,./data/pdfs/addtech_2022_report.pdf,74,0.771676,146 ADDTECH ANNUAL REPORT 2021/2022 ADDTECH...,"0 ```json\n{\n ""KPI_Entries"": [\n {\n ...",addtech_2022_report.pdf,right_only


# Merge with entire dataset

# Notes on Left merge on ReportName, Scope and Year for whole dataset

- only in ground truth, not in results: values beyond 2013 which were not extracted by LLM

In [64]:
# Left merge without page numbers
left_merge = pd.merge(ground_truth, results, how="left", 
                 left_on=["ReportName", "scope_man", "year_man"], 
                 right_on=["report_name_short", "extracted_scope_from_llm", "extracted_year_from_llm"],
                 indicator=True)

len(left_merge)

35618

In [65]:
left_merge.value_counts('_merge')

_merge
both          35614
left_only         4
right_only        0
Name: count, dtype: int64

In [69]:
left_merge[left_merge['_merge'] == 'left_only']

,ReportName,scope_man,year_man,value_man,unit_man,page_man,val_name_man,type_man,ms_comment_man,extracted_year_from_llm_orig,...,page_number_used_by_llm,report_name,page_number_to_llm,page_retrieval_scores,page_texts_to_llm,text_response_from_llm,report_name_short,page_numbers_tried_by_llm,automatic_extraction_tried,_merge
20468,kb home_2019_report.pdf,1,2008,0.0,CO2e in metric tons,67,Scope 1,Table,None,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
20469,kb home_2019_report.pdf,1,2009,0.0,CO2e in metric tons,67,Scope 1,Table,None,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
20528,kb home_2019_report.pdf,2lb,2008,42204.0,CO2e in metric tons,67,Estimated Scope 2 (with Scope 1 = 0) greenhous...,Table,None,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
20529,kb home_2019_report.pdf,2lb,2009,20296.0,CO2e in metric tons,67,Estimated Scope 2 (with Scope 1 = 0) greenhous...,Table,None,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


# Notes on Outer merge for whole dataset

- only in ground truth on outer merge same as only in ground truth in left merge: kbhome for 2008 and 2009
- only in results on outer merge, not in ground truth: scope-year combinations which were queried (2010, 2011, 2024, 2025) outside the scope of the ground truth range 2013-2023, mostly NA

In [66]:
# Outer merge without page numbers
outer_merge = pd.merge(ground_truth, results, how="outer", 
                       left_on=["ReportName", "scope_man", "year_man"],
                       right_on=["report_name_short", "extracted_scope_from_llm", "extracted_year_from_llm"],
                       indicator=True)

len(outer_merge)

56060

In [67]:
outer_merge['_merge'].value_counts()

_merge
both          35614
right_only    20442
left_only         4
Name: count, dtype: int64

In [70]:
outer_merge[outer_merge['_merge'] == 'left_only']

,ReportName,scope_man,year_man,value_man,unit_man,page_man,val_name_man,type_man,ms_comment_man,extracted_year_from_llm_orig,...,page_number_used_by_llm,report_name,page_number_to_llm,page_retrieval_scores,page_texts_to_llm,text_response_from_llm,report_name_short,page_numbers_tried_by_llm,automatic_extraction_tried,_merge
32145,kb home_2019_report.pdf,1,2008.0,0.0,CO2e in metric tons,67,Scope 1,Table,None,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
32146,kb home_2019_report.pdf,1,2009.0,0.0,CO2e in metric tons,67,Scope 1,Table,None,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
32217,kb home_2019_report.pdf,2lb,2008.0,42204.0,CO2e in metric tons,67,Estimated Scope 2 (with Scope 1 = 0) greenhous...,Table,None,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
32218,kb home_2019_report.pdf,2lb,2009.0,20296.0,CO2e in metric tons,67,Estimated Scope 2 (with Scope 1 = 0) greenhous...,Table,None,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [71]:
outer_merge[outer_merge['_merge'] == 'right_only']

,ReportName,scope_man,year_man,value_man,unit_man,page_man,val_name_man,type_man,ms_comment_man,extracted_year_from_llm_orig,...,page_number_used_by_llm,report_name,page_number_to_llm,page_retrieval_scores,page_texts_to_llm,text_response_from_llm,report_name_short,page_numbers_tried_by_llm,automatic_extraction_tried,_merge
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010.0,...,47,./data/pdfs/Allianz_2022_report.pdf,47,0.779139,01 Introduction \nand strategy 02 Measuring an...,"0 ```json\n{\n ""KPI_Entries"": []\n}\n```\...",Allianz_2022_report.pdf,"[47, 48, 49, 78, 79, 80, 90, 91, 92]",True,right_only
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010.0,...,48,./data/pdfs/Allianz_2022_report.pdf,48,0.817664,01 Introduction \nand strategy 02 Measuring an...,"0 ```json\n{\n ""KPI_Entries"": []\n}\n```\...",Allianz_2022_report.pdf,"[47, 48, 49, 78, 79, 80, 90, 91, 92]",True,right_only
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010.0,...,49,./data/pdfs/Allianz_2022_report.pdf,49,0.806163,01 Introduction \nand strategy 02 Measuring an...,"0 ```json\n{\n ""KPI_Entries"": []\n}\n```\...",Allianz_2022_report.pdf,"[47, 48, 49, 78, 79, 80, 90, 91, 92]",True,right_only
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010.0,...,80,./data/pdfs/Allianz_2022_report.pdf,80,0.793936,01 Introduction \nand strategy\n8002 Measuring...,"0 ```json\n{\n ""KPI_Entries"": []\n}\n```\...",Allianz_2022_report.pdf,"[47, 48, 49, 78, 79, 80, 90, 91, 92]",True,right_only
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010.0,...,90,./data/pdfs/Allianz_2022_report.pdf,90,0.765333,01 Introduction \nand strategy02 Measuring and...,"0 ```json\n{\n ""KPI_Entries"": []\n}\n```\...",Allianz_2022_report.pdf,"[47, 48, 49, 78, 79, 80, 90, 91, 92]",True,right_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56055,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025.0,...,90,./data/pdfs/xvivo perfusion_2021_report.pdf,90,0.746226,Note 8. Auditor’s fees and reimbursement of co...,"0 ```json\n{\n ""KPI_Entries"": []\n}\n```\...",xvivo perfusion_2021_report.pdf,"[89, 90, 91, 92, 93, 94]",True,right_only
56056,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025.0,...,91,./data/pdfs/xvivo perfusion_2021_report.pdf,91,0.745386,"Cash flow disclosures, leases Group\n2021 2020...","0 ```json\n{\n ""KPI_Entries"": []\n}\n```\...",xvivo perfusion_2021_report.pdf,"[89, 90, 91, 92, 93, 94]",True,right_only
56057,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025.0,...,92,./data/pdfs/xvivo perfusion_2021_report.pdf,92,0.728455,Note 11. Net financial income\nGroup Parent Co...,"0 ```json\n{\n ""KPI_Entries"": []\n}\n```\...",xvivo perfusion_2021_report.pdf,"[89, 90, 91, 92, 93, 94]",True,right_only
56058,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025.0,...,93,./data/pdfs/xvivo perfusion_2021_report.pdf,93,0.753800,Tax attributable to other comprehensive income...,"0 ```json\n{\n ""KPI_Entries"": []\n}\n```\...",xvivo perfusion_2021_report.pdf,"[89, 90, 91, 92, 93, 94]",True,right_only


# Reports with duplicates

In [45]:
# Reports with double entries
reports_to_exclude = ['Allianz_2022_report.pdf',
                    'acuity brands inc_2022_report.pdf',
                    'aixtron_2020_report.pdf',
                    'allfunds group_2021_report.pdf',
                    'apollo commercial real estate fina_2019_report.pdf',
                    'autoneum holding_2019_report.pdf',
                    'chase corp_2020_report.pdf',
                    'cubesmart reit_2021_report.pdf',
                    'dksh holding_2021_report.pdf',
                    'granite construction inc_2020_report.pdf',
                    'independence realty inc_2017_report.pdf',
                    'jetblue airways corp_2019_report.pdf',
                    'kb home_2019_report.pdf',
                    'sumitomo warehouse ltd_2021_report.pdf',
                    'vital energy inc_2019_report.pdf']
ground_truth_subset = ground_truth[~ground_truth['ReportName'].isin(
    reports_to_exclude)]

len(ground_truth_subset)

4962

In [76]:
ground_truth[ground_truth['ReportName'].isin(reports_to_exclude)]

,ReportName,scope_man,year_man,value_man,unit_man,page_man,val_name_man,type_man,ms_comment_man
0,Allianz_2022_report.pdf,1,2013,NaN,NaN,NaN,NaN,NaN,None
1,Allianz_2022_report.pdf,1,2014,NaN,NaN,NaN,NaN,NaN,None
2,Allianz_2022_report.pdf,1,2015,NaN,NaN,NaN,NaN,NaN,None
3,Allianz_2022_report.pdf,1,2016,NaN,NaN,NaN,NaN,NaN,None
4,Allianz_2022_report.pdf,1,2017,NaN,NaN,NaN,NaN,NaN,None
...,...,...,...,...,...,...,...,...,...
5479,vital energy inc_2019_report.pdf,3,2019,8.7,million metric tons CO2,16,Scope 3 indirect emissions related to extracti...,Text,None
5480,vital energy inc_2019_report.pdf,3,2019,8.7,m metric t CO2,67,Upstream indirect Scope 3 CO2 emissions,Table,None
5481,vital energy inc_2019_report.pdf,3,2020,NaN,NaN,NaN,NaN,NaN,None
5482,vital energy inc_2019_report.pdf,3,2021,NaN,NaN,NaN,NaN,NaN,None


## Duplicate patterns

15 reports concerned

| Pattern | Description | Example | Occurence (scope-year combinations) | Reports |
| ------- | ----------- | ------------- | -------- | -------- |
| Different unit scales, different pages | The same scope-year combination displayed with different unit scales on different pages| Allianz, 2020 1 28714tCO2e (p. 78) vs. 29 ktCO2e (p.92) | 9+2=11 | Allianz, vital | 
| Different units, different pages | The same scope-year combination displayed with different units on different pages, could be normalized | acuity, 2022 1 34327 MtCO2e (p.101) vs. MT CO2e (p.99) | 2+3+2+4+2+2+1+1=17 | acuity, aixtron, allfunds, dksh, independence, jetblue, kb home, vital |
| Same units, different pages | The same scope-year combination displayed with same units on different pages | chase, 69570 MtCO2e (p. 29) and (p. 32) | 2+7+1+15+1+3=29 | chase, dksh, granite, jetblue, kb home, vital  |
| Rounded vs. not rounded value, different pages | The same scope-year combination displayed with different values, same units, on different pages | granite, 244727.0 vs. 244727.61 CO2 Equivalents (US Tons) (p.*66 vs. p.*67) | 1 | granite |

**Accidental duplicates (need to be deleted):**
- apollo commerical real estate: 2019 1 duplicate, but second row all NA
- autoneum: 2018 1; 2019 1
- cubesmart: 2020 1
- sumitomo: 2020 1
- Allianz: 2019 2mb, 2019 3
- allfunds: 2019 1, 2020 1, 2019 2lb, 2020 2lb, 
- apollo commercial: 2019 2lb
- cubesmart: 2021 1
- 

metric tonnes of CO2 equivalent

### Solutions

1. Normalize units before evaluation
2. Find a simple way to treat duplicates

Column indicating potential duplicate, even before merge
- based on page_retrieval_score
- based on confidence score (log probs)
- based on other values/units from same report (also applicable to gold standard)
- Is correct page also the one with highest retrieval score?

Filtering step before merge:
- Prioritize within duplicates using other values/units from same report; standard unit?

Do we want to count the duplicates twice? 
- merge pages into list of pages?


#### Decision
1. Normalize units before computing error categories.
2. Choose entry of page from majority voting.

Same procedure for gold standard and LLM output!


# Questions to resolve

1. What do we do with duplicate entries?

    a) different units
    
    b) same value and unit on different pages

2. How do we merge? Left/Outer with/without page? 

==> Left merge without page

## Which combinations do we want to measure?

1. Value in ground truth = Value in results and pages coincide (either NA or non-NA)
2. Value in ground truth = Value in results, but wrong page
3. Value in results different from ground truth:

        a) value in results = NA, value in ground truth != NA
        b) value in results != NA & != value in ground truth
        c) value in != NA, value in ground truth = NA

TODOs:
automatic_extraction_tried == reports queried?
page_numbers_tried_by_llm : set to NA in merge?

In [ ]:
results[results['automatic_extraction_tried'] == False] # automatic extraction tried for all scope-year combinations

len(results["report_name_short"].unique()) # results for 139 reports

139

In [87]:
merge_without_column_copy = pd.merge(ground_truth, results, how="left",
                             left_on=["ReportName",
                                    "scope_man", "year_man"],
                            right_on=["report_name_short", "extracted_scope_from_llm",
                                    "extracted_year_from_llm"], indicator=True)

In [88]:
merge_without_column_copy["_merge"].value_counts()

_merge
both          35614
left_only         4
right_only        0
Name: count, dtype: int64

In [84]:
merge_without_column_copy[merge_without_column_copy['automatic_extraction_tried'].isna()]

,ReportName,scope_man,year_man,value_man,unit_man,page_man,val_name_man,type_man,ms_comment_man,extracted_year_from_llm_orig,...,raw_llm_response,page_number_used_by_llm,report_name,page_number_to_llm,page_retrieval_scores,page_texts_to_llm,text_response_from_llm,report_name_short,page_numbers_tried_by_llm,automatic_extraction_tried
20468,kb home_2019_report.pdf,1,2008,0.0,CO2e in metric tons,67,Scope 1,Table,None,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20469,kb home_2019_report.pdf,1,2009,0.0,CO2e in metric tons,67,Scope 1,Table,None,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20528,kb home_2019_report.pdf,2lb,2008,42204.0,CO2e in metric tons,67,Estimated Scope 2 (with Scope 1 = 0) greenhous...,Table,None,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20529,kb home_2019_report.pdf,2lb,2009,20296.0,CO2e in metric tons,67,Estimated Scope 2 (with Scope 1 = 0) greenhous...,Table,None,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


For scope-year combinations only found in ground truth all original columns of results dataframe are set to missing, incl. report_name_short, automatic_extraction_tried etc.
To use these rows for evaluation later would need to manually copy over entries such as ReportName, scope_man, year_man etc.
Instead we have decided to copy some columns to ground truth so that the merge does not produce NAs.

In [ ]:
# Create additional variables "automatic_extraction_tried" and "page_numbers_tried_by_llm" in ground_truth
grtruth_extended = results[[
    "report_name_short", "automatic_extraction_tried",  "page_numbers_tried_by_llm"]]

# Subset only one report for testing
grtruth_extended = grtruth_extended[grtruth_extended['report_name_short'] == 'Allianz_2022_report.pdf']

# Transform page numbers from list to tuple for grouping
grtruth_extended['page_numbers_tried_by_llm'] = grtruth_extended['page_numbers_tried_by_llm'].apply(
    tuple)

# Creates table with unique report names, page numbers tried by LLM and whether automatic extraction was tried (all TRUE)
grtruth_extended = grtruth_extended.groupby(["report_name_short", "page_numbers_tried_by_llm"])[
    'automatic_extraction_tried'].apply(lambda group: group.any(skipna=True)).reset_index()

# Transform page numbers back to list for easier readability
grtruth_extended['page_numbers_tried_by_llm'] = grtruth_extended['page_numbers_tried_by_llm'].apply(
    list)

# Merge the extended ground truth with the original ground truth
# to include the new variables in the ground truth dataframe
grtruth_extended = pd.merge(ground_truth, grtruth_extended, how="left",
                            left_on="ReportName", right_on="report_name_short")

# Delete duplicate column 'report_name_short'
grtruth_extended = grtruth_extended.drop('report_name_short', axis=1)


In [ ]:
grtruth_extended[grtruth_extended['automatic_extraction_tried'].isna()] 

,ReportName,scope_man,year_man,value_man,unit_man,page_man,val_name_man,type_man,ms_comment_man,page_numbers_tried_by_llm,automatic_extraction_tried
51,Daimler_2020_report.pdf,1,2013,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN
52,Daimler_2020_report.pdf,1,2014,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN
53,Daimler_2020_report.pdf,1,2015,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN
54,Daimler_2020_report.pdf,1,2016,1056.0,thousand tonnes,63,CO2 direct (Scope 1),Table,None,NaN,NaN
55,Daimler_2020_report.pdf,1,2017,1192.0,thousand tonnes,63,CO2 direct (Scope 1),Table,None,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
5639,xvivo perfusion_2021_report.pdf,3,2018,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN
5640,xvivo perfusion_2021_report.pdf,3,2019,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN
5641,xvivo perfusion_2021_report.pdf,3,2020,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN
5642,xvivo perfusion_2021_report.pdf,3,2021,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN


In [ ]:
# Fill missing values in 'automatic_extraction_tried' with False
# Particularly relevant if only a subset of reports was queried
grtruth_extended.loc[grtruth_extended['automatic_extraction_tried'].isna(
), 'automatic_extraction_tried'] = False


In [ ]:
# 2. Strip irrelevant rows from LLM output
# it is not obvious how to keep rows that have "helpful" data/ extracted values.
# at this step we lose information about page numbers passed to the LLM
# if no valuable information was returned.
#  Option 1:
tiny_results = results[results['extracted_value_from_llm'].notnull(
)]

# Option 2:
# tiny_results = co2_emission_table2_w_query_responses[~(
#            (co2_emission_table2_w_query_responses['extracted_value_from_llm_orig'] == "Not specified") |
#            (co2_emission_table2_w_query_responses['extracted_value_from_llm_orig'] == "Nothing extracted. No Regex match")
# )]

# 3. Left-merge
tiny_results = tiny_results.drop(
    'automatic_extraction_tried', axis=1)
tiny_results = tiny_results.drop(
    'page_numbers_tried_by_llm', axis=1)

In [ ]:
merged_results = pd.merge(grtruth_extended, tiny_results, how="left",
                            left_on=["ReportName",
                                    "scope_man", "year_man"],
                            right_on=["report_name_short", "extracted_scope_from_llm",
                                    "extracted_year_from_llm"])